In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data_loader import load_pjm_data

df = load_pjm_data("../data/raw/PJME_hourly.csv")

# Same 2016-2018 subset we've been using for ML models, for fair comparison
df_lstm = df.loc["2016-01-01":].copy()
print(df_lstm.shape)

(22681, 1)


In [2]:
split_idx = int(len(df_lstm) * 0.85)

train_data = df_lstm.iloc[:split_idx]
test_data = df_lstm.iloc[split_idx:]

print(f"Train: {train_data.index.min()} to {train_data.index.max()} ({len(train_data)} rows)")
print(f"Test: {test_data.index.min()} to {test_data.index.max()} ({len(test_data)} rows)")

Train: 2016-01-01 00:00:00 to 2018-03-14 05:00:00 (19278 rows)
Test: 2018-03-14 06:00:00 to 2018-08-03 00:00:00 (3403 rows)


In [3]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))

train_scaled = scaler.fit_transform(train_data[["PJME_MW"]])
test_scaled = scaler.transform(test_data[["PJME_MW"]])

print(train_scaled[:5].flatten())
print(f"Min: {train_scaled.min()}, Max: {train_scaled.max()}")

[0.20010233 0.16840801 0.14390349 0.12497307 0.11611374]
Min: 0.0, Max: 1.0


In [4]:
def create_sequences(data, n_steps):
    X, y = [], []
    for i in range(n_steps, len(data)):
        X.append(data[i - n_steps:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

n_steps = 48  # look back 48 hours (2 days)

X_train, y_train = create_sequences(train_scaled, n_steps)
X_test, y_test = create_sequences(test_scaled, n_steps)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (19230, 48)
y_train shape: (19230,)


In [5]:
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("X_train shape (samples, timesteps, features):", X_train.shape)

X_train shape (samples, timesteps, features): (19230, 48, 1)


In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model_lstm = Sequential([
    LSTM(64, return_sequences=True, input_shape=(n_steps, 1)),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(1)
])

model_lstm.compile(optimizer="adam", loss="mean_squared_error")
model_lstm.summary()

c:\Users\hp\Desktop\Machine Learning\Energy-Forecast\.conda\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 48, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 48, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,345 (114.63 KB)

 Trainable params: 29,345 (114.63 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
history = model_lstm.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

Epoch 1/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 26s 37ms/step - loss: 0.0086 - val_loss: 0.0035
Epoch 2/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 20s 37ms/step - loss: 0.0033 - val_loss: 0.0024
Epoch 3/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 22s 41ms/step - loss: 0.0021 - val_loss: 9.2652e-04
Epoch 4/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 23s 42ms/step - loss: 0.0014 - val_loss: 6.0800e-04
Epoch 5/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 19s 36ms/step - loss: 0.0011 - val_loss: 6.7773e-04
Epoch 6/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 20s 37ms/step - loss: 9.0562e-04 - val_loss: 4.0384e-04
Epoch 7/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 20s 38ms/step - loss: 7.8736e-04 - val_loss: 3.3881e-04
Epoch 8/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 20s 36ms/step - loss: 7.0690e-04 - val_loss: 5.3947e-04
Epoch 9/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 20s 36ms/step - loss: 6.5553e-04 - val_loss: 3.0259e-04
Epoch 10/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 20s 36ms/step - loss: 5.9603e-04 - val_loss: 2.3583e-04
Epoch 11/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 19s 36ms/step - loss: 5.95

In [8]:
predictions_scaled = model_lstm.predict(X_test)

# Un-scale back to actual MW values
predictions = scaler.inverse_transform(predictions_scaled)
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

105/105 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step


In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae_lstm = mean_absolute_error(y_test_actual, predictions)
rmse_lstm = np.sqrt(mean_squared_error(y_test_actual, predictions))

print(f"LSTM Test MAE: {mae_lstm:.2f}")
print(f"LSTM Test RMSE: {rmse_lstm:.2f}")

LSTM Test MAE: 347.32
LSTM Test RMSE: 440.89
